# Rush Hour Radar: NYC taxi demand forecasting

Hourly yellow-taxi pickup forecasts for the 20 busiest NYC zones, a day ahead, with 80% prediction intervals.

**Data:** NYC TLC Trip Record Data (yellow taxi, 2023 and 2024), downloaded directly in the notebook.
**Run:** designed for Google Colab (Drive is used to store intermediate files). Run the cells top to bottom.

| Section | What it does |
|---|---|
| 1-2 | Build the hourly demand table, explore patterns, fix daylight-saving hours |
| 3 | Chronological split and baselines |
| 4-5 | LightGBM point forecast (v1) and 80% quantile band |
| 6 | Extreme-event and anomaly analysis |
| 7 | Final test, v1 (Nov-Dec 2024, run once) |
| 8-9 | v2: add 2023 data and calendar features, final test again |
| 10 | Figures |
| 11 | Weather features (Open-Meteo, hourly) |
| 12 | Rolling backtest: v2 vs v2 + weather, month by month |
| 13 | 80% bands for the backtest and export for the Streamlit dashboard |

## 1. Build the hourly demand table (2024)

In [ ]:
import pandas as pd

base = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{:02d}.parquet"
frames = []
for m in range(1, 13):
    df = pd.read_parquet(base.format(m), columns=["tpep_pickup_datetime", "PULocationID"])
    frames.append(df)
trips = pd.concat(frames, ignore_index=True)

# Drop stray rows dated outside 2024
trips = trips[(trips["tpep_pickup_datetime"] >= "2024-01-01") &
              (trips["tpep_pickup_datetime"] < "2025-01-01")]
print(trips.shape)

trips["hour"] = trips["tpep_pickup_datetime"].dt.floor("h")

top20 = trips["PULocationID"].value_counts().head(20).index
trips = trips[trips["PULocationID"].isin(top20)]

demand = (trips.groupby(["hour", "PULocationID"]).size()
               .rename("trips").reset_index())

# Fill hours with zero trips so every zone has every hour
full = pd.MultiIndex.from_product(
    [pd.date_range("2024-01-01", "2024-12-31 23:00", freq="h"), top20],
    names=["hour", "PULocationID"])
demand = (demand.set_index(["hour", "PULocationID"])
                .reindex(full, fill_value=0).reset_index())

print(demand.shape)
demand.head()

print(demand["trips"].describe())
print(demand.isna().sum())

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/rush-hour-radar", exist_ok=True)
demand.to_parquet("/content/drive/MyDrive/rush-hour-radar/demand.parquet")
demand = pd.read_parquet("/content/drive/MyDrive/rush-hour-radar/demand.parquet")

## 2. Explore the data and fix daylight-saving hours

In [ ]:
import matplotlib.pyplot as plt

total = demand.groupby("hour")["trips"].sum()

fig, ax = plt.subplots(2, 1, figsize=(12, 7))
total["2024-03-04":"2024-03-17"].plot(ax=ax[0], title="Total demand, two weeks in March")
hourly_avg = demand.assign(h=demand["hour"].dt.hour).groupby("h")["trips"].mean()
hourly_avg.plot(ax=ax[1], title="Average demand by hour of day")
plt.tight_layout()
plt.show()

In [ ]:
dst_hours = pd.to_datetime(["2024-03-10 02:00", "2024-11-03 01:00"])
print(demand[demand["hour"].isin(dst_hours)].groupby("hour")["trips"].sum())

In [ ]:
import pandas as pd

demand["hour"] = pd.to_datetime(demand["hour"])
tot = demand.groupby("hour")["trips"].sum()

# 1) Confirm DST diagnosis: compare to same hour on neighbouring Sundays
dst_hours = pd.to_datetime(["2024-03-10 02:00", "2024-11-03 01:00"])
for t in dst_hours:
    nbrs = [t - pd.Timedelta(days=7), t + pd.Timedelta(days=7)]
    print(t, "->", tot[t], "| Sundays before/after:", [tot[n] for n in nbrs])

# 2) Clean table for modeling
dc = demand[~demand["hour"].isin(dst_hours)].copy()
print(dc.shape)   # expect (175640, 3)

# 3) Any other zeros? (real zero-trip hours vs. missing data)
zeros = dc[dc["trips"] == 0]
print(len(zeros), "zero rows")
print(zeros.groupby("PULocationID").size().sort_values(ascending=False).head())

In [ ]:
# 4) Features for plotting
dc["h"] = dc["hour"].dt.hour
dc["dow"] = dc["hour"].dt.dayofweek          # 0 = Mon
dc["weekend"] = dc["dow"] >= 5

# 5) Daily pattern, weekday vs weekend
daily = dc.groupby(["weekend", "h"])["trips"].mean().unstack(0)
daily.columns = ["Weekday", "Weekend"]
ax = daily.plot(figsize=(9, 4), marker="o", title="Mean trips per zone-hour")
ax.set_xlabel("Hour of day"); ax.set_xticks(range(0, 24, 2)); ax.grid(alpha=0.3)
plt.show()

# 6) Weekly heatmap
heat = dc.groupby(["dow", "h"])["trips"].mean().unstack()
plt.figure(figsize=(11, 3.5))
plt.imshow(heat, aspect="auto", cmap="viridis")
plt.yticks(range(7), ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
plt.xticks(range(0, 24, 2), range(0, 24, 2))
plt.colorbar(label="Mean trips"); plt.title("Demand by day of week and hour")
plt.xlabel("Hour of day"); plt.show()

# 7) Do zones differ in level, shape, or both?
top4 = dc.groupby("PULocationID")["trips"].sum().nlargest(4).index
fig, ax = plt.subplots(figsize=(9, 4))
for z in top4:
    s = dc[dc["PULocationID"] == z].groupby("h")["trips"].mean()
    ax.plot(s.index, s.values, label=f"Zone {z}")
ax.set_title("Daily shape, 4 busiest zones"); ax.set_xlabel("Hour"); ax.legend()
plt.show()

## 3. Chronological split and baselines

In [ ]:
import numpy as np
import pandas as pd

dc = dc.sort_values(["PULocationID", "hour"]).reset_index(drop=True)

# Lags by timestamp merge (NOT shift), so the 2 removed DST hours don't misalign rows
def add_lag(df, hours, name):
    lag = df[["hour", "PULocationID", "trips"]].copy()
    lag["hour"] = lag["hour"] + pd.Timedelta(hours=hours)
    lag = lag.rename(columns={"trips": name})
    return df.merge(lag, on=["hour", "PULocationID"], how="left")

dc = add_lag(dc, 24, "lag_24")
dc = add_lag(dc, 168, "lag_168")

# Chronological split
train = dc[dc["hour"] < "2024-10-01"].copy()
val   = dc[(dc["hour"] >= "2024-10-01") & (dc["hour"] < "2024-11-01")].copy()
test  = dc[dc["hour"] >= "2024-11-01"].copy()      # locked until the final evaluation
print(len(train), len(val), len(test))

# Baseline 3: hour-of-week average per zone, learned from train only
for d in (train, val):
    d["h"] = d["hour"].dt.hour
    d["dow"] = d["hour"].dt.dayofweek
how = (train.groupby(["PULocationID", "dow", "h"])["trips"]
            .mean().rename("how_mean").reset_index())
val = val.merge(how, on=["PULocationID", "dow", "h"], how="left")

# Metrics (WAPE instead of MAPE, because zeros make MAPE blow up)
def scores(y, p):
    m = ~(np.isnan(y) | np.isnan(p))
    y, p = y[m], p[m]
    return {"MAE": np.abs(y - p).mean(),
            "RMSE": np.sqrt(((y - p) ** 2).mean()),
            "WAPE": np.abs(y - p).sum() / y.sum()}

y = val["trips"].values
results = pd.DataFrame({
    "same hour yesterday": scores(y, val["lag_24"].values),
    "same hour last week": scores(y, val["lag_168"].values),
    "hour-of-week mean":   scores(y, val["how_mean"].values),
}).T
print(results.round(3))

## 4. LightGBM v1 (point forecast)

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

# --- features (all lags >= 24h, so day-ahead safe) ---
dc["h"] = dc["hour"].dt.hour
dc["dow"] = dc["hour"].dt.dayofweek
dc["zone"] = dc["PULocationID"].astype("category")        # fixed categories across splits

for k, name in [(336, "lag_336"), (504, "lag_504"), (672, "lag_672")]:
    dc = add_lag(dc, k, name)
dc["lag_wk_mean"] = dc[["lag_168", "lag_336", "lag_504", "lag_672"]].mean(axis=1)

hol = USFederalHolidayCalendar().holidays(start="2023-12-01", end="2025-01-31")
dc["is_holiday"] = dc["hour"].dt.normalize().isin(hol).astype(int)

FEATURES = ["zone", "h", "dow", "is_holiday",
            "lag_24", "lag_168", "lag_336", "lag_wk_mean"]

train = dc[dc["hour"] < "2024-10-01"]
val   = dc[(dc["hour"] >= "2024-10-01") & (dc["hour"] < "2024-11-01")]
# test stays untouched

In [ ]:
# --- model ---
model = lgb.LGBMRegressor(
    objective="poisson",          # counts, always >= 0
    n_estimators=2000, learning_rate=0.03,
    num_leaves=31, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1)

model.fit(train[FEATURES], train["trips"],
          eval_set=[(val[FEATURES], val["trips"])],
          eval_metric="l1",
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

val = val.assign(pred=model.predict(val[FEATURES]))

# --- compare with baselines (reuses scores() and results from last cell) ---
results.loc["LightGBM"] = scores(val["trips"].values, val["pred"].values)
print(results.round(3))
print("best iteration:", model.best_iteration_)

# --- where does it still fail? ---
val["abs_err"] = (val["trips"] - val["pred"]).abs()
print(val.groupby("zone", observed=True)["abs_err"].mean().sort_values(ascending=False).head(5))
print(val.groupby("h")["abs_err"].mean().round(1).to_string())

imp = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

## 5. Prediction intervals (80% band)

In [ ]:
val = val.copy()
qmodels = {}
for a in (0.1, 0.9):
    m = lgb.LGBMRegressor(
        objective="quantile", alpha=a,
        n_estimators=2000, learning_rate=0.03,
        num_leaves=31, min_child_samples=50,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, verbose=-1)
    m.fit(train[FEATURES], train["trips"],
          eval_set=[(val[FEATURES], val["trips"])],
          eval_metric="quantile",
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    qmodels[a] = m
    val[f"q{int(a*100)}"] = m.predict(val[FEATURES])

# keep the band sensible: lower <= point forecast <= upper, lower >= 0
val["lo"] = np.maximum(0, np.minimum(val["q10"], val["pred"]))
val["hi"] = np.maximum(val["q90"], val["pred"])

val["covered"] = (val["trips"] >= val["lo"]) & (val["trips"] <= val["hi"])
val["width"] = val["hi"] - val["lo"]

print("overall coverage (target 0.80):", val["covered"].mean().round(3))
print("mean width:", val["width"].mean().round(1),
      "| relative width:", (val["width"].sum() / val["trips"].sum()).round(3))

print("\ncoverage by hour")
print(val.groupby("h")["covered"].mean().round(2).to_string())

print("\ncoverage by zone")
print(val.groupby("zone", observed=True)["covered"].mean().round(2).sort_values().head(6))

print("\ncoverage weekday vs weekend")
print(val.groupby(val["dow"] >= 5)["covered"].mean().round(3))

# visual check: one busy zone, one week
z = val["zone"].value_counts().index[0]
w = val[(val["zone"] == z) & (val["hour"] >= "2024-10-07") & (val["hour"] < "2024-10-14")]
plt.figure(figsize=(12, 4))
plt.fill_between(w["hour"], w["lo"], w["hi"], alpha=0.25, label="80% band")
plt.plot(w["hour"], w["trips"], label="actual", lw=1.5)
plt.plot(w["hour"], w["pred"], label="forecast", lw=1, ls="--")
plt.title(f"Zone {z}, Oct 7 to 13 (includes Columbus Day, Oct 14 is next)")
plt.legend(); plt.show()

## 6. Extreme events and anomalies

In [ ]:
# hours where actual demand fell outside the band
val["above"] = val["trips"] > val["hi"]
val["below"] = val["trips"] < val["lo"]
val["miss_size"] = np.where(val["above"], val["trips"] - val["hi"],
                     np.where(val["below"], val["lo"] - val["trips"], 0))

# worst misses
worst = val.sort_values("miss_size", ascending=False).head(15)
print(worst[["hour", "PULocationID", "trips", "lo", "pred", "hi", "miss_size"]].round(0))

# which dates have the most misses overall?
by_day = (val.assign(day=val["hour"].dt.date)
             .groupby("day")["miss_size"].sum()
             .sort_values(ascending=False).head(8))
print(by_day.round(0))

# total demand across all zones per day vs. forecast
day = val.assign(day=val["hour"].dt.date).groupby("day")[["trips", "pred"]].sum()
day["error_pct"] = ((day["trips"] - day["pred"]) / day["pred"] * 100).round(1)
print(day.sort_values("error_pct").head(5))
print(day.sort_values("error_pct").tail(5))

In [ ]:
# 1) how far outside the band, measured in "band widths"
val["band"] = (val["hi"] - val["lo"]).clip(lower=1)
val["miss_ratio"] = val["miss_size"] / val["band"]      # 0 = inside the band
val["extreme"] = val["miss_ratio"] > 1.0                # more than 1 full band-width outside
print("extreme hours:", val["extreme"].sum(), f"({val['extreme'].mean():.2%} of all hours)")
print("too high vs too low:", val.loc[val["extreme"], "above"].value_counts().to_dict())

# 2) sustained = 3+ outside-band hours in a row in the same zone
val = val.sort_values(["PULocationID", "hour"]).reset_index(drop=True)
outside = (val["above"] | val["below"]).astype(int)
same_zone = val["PULocationID"].eq(val["PULocationID"].shift())
grp = (~(outside.eq(outside.shift()) & same_zone)).cumsum()      # new group when status or zone changes
val["run_len"] = outside.groupby(grp).transform("sum") * outside
val["sustained"] = val["run_len"] >= 3

# 3) which days and zones have the most flags?
by_day = (val.groupby(val["hour"].dt.date)[["extreme", "sustained"]].sum()
             .sort_values("extreme", ascending=False).head(8))
print(by_day)
print(val[val["sustained"]].groupby("PULocationID").size().sort_values(ascending=False).head())

In [ ]:
d = val.assign(day=val["hour"].dt.date)
day = d.groupby("day").agg(
    pct_above=("above", "mean"),
    pct_below=("below", "mean"),
    actual=("trips", "sum"),
    forecast=("pred", "sum"))
day["error_pct"] = ((day["actual"] - day["forecast"]) / day["forecast"] * 100).round(1)
day["citywide_flag"] = (day["pct_above"] > 0.30) | (day["pct_below"] > 0.30)
print(day.round(2).sort_values("error_pct").to_string())
print("flagged days:", day.index[day["citywide_flag"]].tolist())

In [ ]:
# save v1 artifacts to Drive
out = "/content/drive/MyDrive/rush-hour-radar"
dc.to_parquet(f"{out}/features_clean.parquet")
val.to_parquet(f"{out}/val_predictions.parquet")
model.booster_.save_model(f"{out}/lgbm_point.txt")
for a, m in qmodels.items():
    m.booster_.save_model(f"{out}/lgbm_q{int(a*100)}.txt")

## 7. Final test, v1 (Nov-Dec 2024, run once)

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb

# 0) sanity checks before touching the test set
print("columns:", [c for c in dc.columns if c.endswith(("_x", "_y"))] or "no duplicate columns")
print("train:", (dc["hour"] < "2024-10-01").sum(),
      "| val:", ((dc["hour"] >= "2024-10-01") & (dc["hour"] < "2024-11-01")).sum(),
      "| test:", (dc["hour"] >= "2024-11-01").sum())     # expect 131500 / 14880 / 29260


# 2) final test run (run ONCE, don't tune afterwards)
def scores(y, p):
    m = ~(np.isnan(y) | np.isnan(p)); y, p = y[m], p[m]
    return {"MAE": np.abs(y - p).mean(),
            "RMSE": np.sqrt(((y - p) ** 2).mean()),
            "WAPE": np.abs(y - p).sum() / y.sum()}

BASE = dict(learning_rate=0.03, num_leaves=31, min_child_samples=50,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=42, verbose=-1)

trval = dc[dc["hour"] < "2024-11-01"]
test  = dc[dc["hour"] >= "2024-11-01"].copy()

how = (trval.groupby(["PULocationID", "dow", "h"])["trips"]
            .mean().rename("how_mean").reset_index())
test = test.merge(how, on=["PULocationID", "dow", "h"], how="left")

point = lgb.LGBMRegressor(objective="poisson", n_estimators=1096, **BASE
        ).fit(trval[FEATURES], trval["trips"])
q10 = lgb.LGBMRegressor(objective="quantile", alpha=0.1, n_estimators=1998, **BASE
        ).fit(trval[FEATURES], trval["trips"])
q90 = lgb.LGBMRegressor(objective="quantile", alpha=0.9, n_estimators=545, **BASE
        ).fit(trval[FEATURES], trval["trips"])

test["pred"] = point.predict(test[FEATURES])
test["lo"] = np.maximum(0, np.minimum(q10.predict(test[FEATURES]), test["pred"]))
test["hi"] = np.maximum(q90.predict(test[FEATURES]), test["pred"])

y = test["trips"].values
res = pd.DataFrame({
    "same hour yesterday": scores(y, test["lag_24"].values),
    "same hour last week": scores(y, test["lag_168"].values),
    "hour-of-week mean":   scores(y, test["how_mean"].values),
    "LightGBM":            scores(y, test["pred"].values)}).T
print(res.round(3))
res_v1 = res.copy()          # keep v1 results for the v2 comparison

test["covered"] = (test["trips"] >= test["lo"]) & (test["trips"] <= test["hi"])
cov_v1 = test["covered"].mean()
print("coverage (target 0.80):", test["covered"].mean().round(3),
      "| mean width:", (test["hi"] - test["lo"]).mean().round(1))
print(test.groupby(test["hour"].dt.month)["covered"].mean().round(3))

test["day"] = test["hour"].dt.date
day = test.groupby("day")[["trips", "pred"]].sum()
day["error_pct"] = ((day["trips"] - day["pred"]) / day["pred"] * 100).round(1)
print(day.sort_values("error_pct").head(6))
print(day.sort_values("error_pct").tail(6))

## 8. v2: add 2023 data

In [ ]:
import pandas as pd, numpy as np

zones = sorted(demand["PULocationID"].unique())      # same 20 zones as 2024
print(len(zones), "zones")

base = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
parts = []
for m in range(1, 13):
    t = pd.read_parquet(base.format(m), columns=["tpep_pickup_datetime", "PULocationID"])
    t = t[t["PULocationID"].isin(zones)]
    t["hour"] = t["tpep_pickup_datetime"].dt.floor("h")
    t = t[(t["hour"] >= "2023-01-01") & (t["hour"] < "2024-01-01")]   # drop stray timestamps
    parts.append(t.groupby(["hour", "PULocationID"]).size().rename("trips").reset_index())
    print(m, "done")

d23 = pd.concat(parts).groupby(["hour", "PULocationID"], as_index=False)["trips"].sum()

# full grid so zero-trip hours exist as rows
grid = pd.MultiIndex.from_product(
    [pd.date_range("2023-01-01", "2023-12-31 23:00", freq="h"), zones],
    names=["hour", "PULocationID"]).to_frame(index=False)
d23 = grid.merge(d23, on=["hour", "PULocationID"], how="left").fillna({"trips": 0})
d23["trips"] = d23["trips"].astype(int)
print(d23.shape)     # expect (175200, 3) = 8,760 hours x 20 zones

d23.to_parquet("/content/drive/MyDrive/rush-hour-radar/demand_2023.parquet")
print(d23["trips"].describe().round(1))

## 9. v2: two-year model, calendar features and final test

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb
from pandas.tseries.holiday import USFederalHolidayCalendar

out = "/content/drive/MyDrive/rush-hour-radar"

# 1) combine 2023 + 2024, drop the 4 DST hours
raw = pd.concat([pd.read_parquet(f"{out}/demand_2023.parquet"),
                 pd.read_parquet(f"{out}/demand.parquet")[["hour", "PULocationID", "trips"]]])
raw["hour"] = pd.to_datetime(raw["hour"])
dst = pd.to_datetime(["2023-03-12 02:00", "2023-11-05 01:00",
                      "2024-03-10 02:00", "2024-11-03 01:00"])
d2 = raw[~raw["hour"].isin(dst)].copy()
print(d2.shape)                                   # expect (350800, 3)

# 2) lags (timestamp merge, not shift)

for k in (24, 168, 336, 504, 672):
    d2 = add_lag(d2, k, f"lag_{k}")
d2["lag_wk_mean"] = d2[["lag_168", "lag_336", "lag_504", "lag_672"]].mean(axis=1)

# 3) calendar features
hol = USFederalHolidayCalendar().holidays(start="2022-12-01", end="2025-01-31")
day = d2["hour"].dt.normalize()
d2["h"] = d2["hour"].dt.hour
d2["dow"] = d2["hour"].dt.dayofweek
d2["month"] = d2["hour"].dt.month
d2["zone"] = d2["PULocationID"].astype("category")
d2["is_holiday"] = day.isin(hol).astype(int)
d2["hol_eve"] = (day + pd.Timedelta(days=1)).isin(hol).astype(int)
d2["hol_after"] = (day - pd.Timedelta(days=1)).isin(hol).astype(int)
md = d2["hour"].dt.month * 100 + d2["hour"].dt.day
d2["xmas_period"] = ((md >= 1224) | (md <= 101)).astype(int)
d2 = d2.sort_values(["PULocationID", "hour"]).reset_index(drop=True)

F2 = ["zone", "h", "dow", "month", "is_holiday", "hol_eve", "hol_after", "xmas_period",
      "lag_24", "lag_168", "lag_336", "lag_wk_mean"]
d2.to_parquet(f"{out}/features_v2.parquet")

train = d2[d2["hour"] < "2024-10-01"]
val   = d2[(d2["hour"] >= "2024-10-01") & (d2["hour"] < "2024-11-01")]
print("train:", len(train), "| val:", len(val))

In [ ]:
# 4) early stopping on Oct 2024 (to find iteration counts)
BASE = dict(learning_rate=0.03, num_leaves=31, min_child_samples=50, subsample=0.8,
            subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
cb = [lgb.early_stopping(100, verbose=False)]
best = {}
for name, kw, metric in [("point", dict(objective="poisson"), "l1"),
                         ("q10", dict(objective="quantile", alpha=0.1), "quantile"),
                         ("q90", dict(objective="quantile", alpha=0.9), "quantile")]:
    m = lgb.LGBMRegressor(n_estimators=3000, **kw, **BASE)
    m.fit(train[F2], train["trips"], eval_set=[(val[F2], val["trips"])],
          eval_metric=metric, callbacks=cb)
    best[name] = m.best_iteration_
    print(name, "best iteration:", best[name])
    if name == "point":
        print("v2 val MAE:", np.abs(val["trips"] - m.predict(val[F2])).mean().round(2), "(v1 was 20.23)")

In [ ]:
# 5) FINAL test run (once): retrain on everything before Nov 2024
def scores(y, p):
    k = ~(np.isnan(y) | np.isnan(p)); y, p = y[k], p[k]
    return {"MAE": np.abs(y - p).mean(), "RMSE": np.sqrt(((y - p) ** 2).mean()),
            "WAPE": np.abs(y - p).sum() / y.sum()}

trval = d2[d2["hour"] < "2024-11-01"]
test = d2[d2["hour"] >= "2024-11-01"].copy()
print("test rows:", len(test))                    # expect 29260

how = (trval.groupby(["PULocationID", "dow", "h"])["trips"].mean().rename("how_mean").reset_index())
test = test.merge(how, on=["PULocationID", "dow", "h"], how="left")

fit = lambda kw, n: lgb.LGBMRegressor(n_estimators=n, **kw, **BASE).fit(trval[F2], trval["trips"])
point = fit(dict(objective="poisson"), best["point"])
q10 = fit(dict(objective="quantile", alpha=0.1), best["q10"])
q90 = fit(dict(objective="quantile", alpha=0.9), best["q90"])

test["pred"] = point.predict(test[F2])
test["lo"] = np.maximum(0, np.minimum(q10.predict(test[F2]), test["pred"]))
test["hi"] = np.maximum(q90.predict(test[F2]), test["pred"])

y = test["trips"].values
res = pd.DataFrame({
    "same hour last week": scores(y, test["lag_168"].values),
    "hour-of-week mean":   scores(y, test["how_mean"].values),
    "LightGBM v1 (2024 only)": res_v1.loc["LightGBM"].to_dict(),
    "LightGBM v2 (2 years)":   scores(y, test["pred"].values)}).T
print(res.round(3))

test["covered"] = (test["trips"] >= test["lo"]) & (test["trips"] <= test["hi"])
print("coverage v2:", test["covered"].mean().round(3), f"(v1 was {cov_v1:.3f}) | width:",
      (test["hi"] - test["lo"]).mean().round(1))
print(test.groupby("month")["covered"].mean().round(3))

test["day"] = test["hour"].dt.date
dd = test.groupby("day")[["trips", "pred"]].sum()
dd["v2_error_pct"] = ((dd["trips"] - dd["pred"]) / dd["pred"] * 100).round(1)
key = pd.to_datetime(["2024-11-05", "2024-11-28", "2024-12-24", "2024-12-25",
                      "2024-12-31", "2024-12-06"]).date
print(dd.loc[key, ["v2_error_pct"]])
print(dd["v2_error_pct"].abs().mean().round(2), "= mean absolute daily error (v2)")

## 10. Figures

In [ ]:
import matplotlib.pyplot as plt
out = "/content/drive/MyDrive/rush-hour-radar"

# 1) model comparison
r = res["MAE"]
plt.figure(figsize=(8, 4))
plt.barh(r.index, r.values, color=["#bbb", "#bbb", "#7aa6d6", "#1f5fa8"])
for i, v in enumerate(r.values): plt.text(v + 0.5, i, f"{v:.1f}", va="center")
plt.xlabel("MAE (trips per zone-hour), Nov to Dec 2024"); plt.title("Forecast error, lower is better")
plt.tight_layout(); plt.savefig(f"{out}/fig_model_comparison.png", dpi=150); plt.show()

# 2) the 80% band over Christmas week, busiest zone
z = test["zone"].value_counts().index[0]
w = test[(test["zone"] == z) & (test["hour"] >= "2024-12-22") & (test["hour"] < "2024-12-29")]
plt.figure(figsize=(12, 4))
plt.fill_between(w["hour"], w["lo"], w["hi"], alpha=0.25, label="80% band")
plt.plot(w["hour"], w["trips"], label="actual", lw=1.5)
plt.plot(w["hour"], w["pred"], "--", label="forecast", lw=1)
plt.title(f"Zone {z}: Christmas week 2024"); plt.legend()
plt.tight_layout(); plt.savefig(f"{out}/fig_christmas_band.png", dpi=150); plt.show()

# 3) daily error, Nov to Dec
plt.figure(figsize=(12, 3.5))
plt.bar(pd.to_datetime(dd.index), dd["v2_error_pct"], color="#1f5fa8")
plt.axhline(8, ls="--", c="gray"); plt.axhline(-8, ls="--", c="gray")
plt.ylabel("daily error %"); plt.title("Daily forecast error (actual vs forecast), v2")
plt.tight_layout(); plt.savefig(f"{out}/fig_daily_error.png", dpi=150); plt.show()

## 11. Weather features (upgrade)

Adds hourly NYC temperature and precipitation (Open-Meteo archive, free, no key).
This is **observed** weather. In real use a day-ahead forecast would need a *weather forecast*, so
the gain measured here is an upper bound. One weather point is used for the whole city.

In [ ]:
import requests
import pandas as pd

out = "/content/drive/MyDrive/rush-hour-radar"

# Hourly weather for NYC (Central Park area), in local time to match the TLC timestamps.
# Source: Open-Meteo historical archive (free, no key). Observed weather, not a forecast.
frames = []
for year in (2023, 2024):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params=dict(
        latitude=40.78, longitude=-73.97,
        start_date=f"{year}-01-01", end_date=f"{year}-12-31",
        hourly="temperature_2m,precipitation", timezone="America/New_York"))
    r.raise_for_status()
    h = r.json()["hourly"]
    frames.append(pd.DataFrame({"hour": pd.to_datetime(h["time"]),
                                "temp_c": h["temperature_2m"],
                                "precip_mm": h["precipitation"]}))
wx = pd.concat(frames).drop_duplicates("hour").reset_index(drop=True)
wx["wet"] = (wx["precip_mm"] >= 0.1).astype(int)       # any measurable precipitation (rain or snow)
print(wx.shape, wx.isna().sum().to_dict())             # expect about 17,544 rows, no missing values
print(wx[["temp_c", "precip_mm", "wet"]].describe().round(1))
wx.to_parquet(f"{out}/weather_hourly.parquet")

# attach weather to the v2 feature table (same weather for every zone)
d2 = pd.read_parquet(f"{out}/features_v2.parquet")
d3 = d2.merge(wx, on="hour", how="left")
assert len(d3) == len(d2), "merge changed the row count"
print("rows:", len(d3), "| missing weather:", d3["temp_c"].isna().sum(),
      "| share of wet hours:", d3["wet"].mean().round(3))
d3.to_parquet(f"{out}/features_v3.parquet")

## 12. Rolling backtest (upgrade)

Instead of one test window, each month from April to December 2024 is predicted by a model trained
only on earlier data (expanding window, fixed settings, no early stopping). Compares v2 with
v3 (v2 + weather). Takes roughly 15 to 25 minutes. Set `FIRST_TEST_MONTH = "2024-10"` for a quick trial.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb

out = "/content/drive/MyDrive/rush-hour-radar"
d3 = pd.read_parquet(f"{out}/features_v3.parquet")

F_v2 = ["zone", "h", "dow", "month", "is_holiday", "hol_eve", "hol_after", "xmas_period",
        "lag_24", "lag_168", "lag_336", "lag_wk_mean"]
F_v3 = F_v2 + ["temp_c", "precip_mm", "wet"]
MODELS = {"v2": F_v2, "v3": F_v3}

# Fixed settings for every fold (no early stopping, so no fold tunes on its own test month)
BT = dict(objective="poisson", n_estimators=1000, learning_rate=0.05, num_leaves=31,
          min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
          random_state=42, verbose=-1, n_jobs=-1)

def scores(y, p):
    k = ~(np.isnan(y) | np.isnan(p)); y, p = y[k], p[k]
    return {"MAE": np.abs(y - p).mean(), "RMSE": np.sqrt(((y - p) ** 2).mean()),
            "WAPE": np.abs(y - p).sum() / y.sum()}

FIRST_TEST_MONTH = "2024-04"      # set to "2024-10" for a quick trial run
months = pd.period_range(FIRST_TEST_MONTH, "2024-12", freq="M")

rows, preds = [], []
for p in months:
    t0, t1 = p.start_time, (p + 1).start_time
    tr = d3[d3["hour"] < t0]                                   # expanding window: everything before the month
    te = d3[(d3["hour"] >= t0) & (d3["hour"] < t1)].copy()
    how = (tr.groupby(["PULocationID", "dow", "h"])["trips"].mean().rename("how_mean").reset_index())
    te = te.merge(how, on=["PULocationID", "dow", "h"], how="left")
    for name, feats in MODELS.items():
        m = lgb.LGBMRegressor(**BT).fit(tr[feats], tr["trips"])
        te[f"pred_{name}"] = m.predict(te[feats])
    y = te["trips"].values
    for label, col in [("same hour last week", "lag_168"), ("hour-of-week mean", "how_mean"),
                       ("LightGBM v2", "pred_v2"), ("LightGBM v3 (+weather)", "pred_v3")]:
        rows.append({"month": str(p), "model": label, **scores(y, te[col].values)})
    preds.append(te[["hour", "PULocationID", "trips", "lag_168", "how_mean",
                     "pred_v2", "pred_v3", "wet", "precip_mm", "temp_c"]])
    print(p, "done | train rows:", len(tr), "| test rows:", len(te))

bt = pd.DataFrame(rows)
pooled = pd.concat(preds, ignore_index=True)

print("\nMAE by month")
print(bt.pivot(index="month", columns="model", values="MAE").round(2))

print("\nPooled over all test months")
cols = {"same hour last week": "lag_168", "hour-of-week mean": "how_mean",
        "LightGBM v2": "pred_v2", "LightGBM v3 (+weather)": "pred_v3"}
print(pd.DataFrame({k: scores(pooled["trips"].values, pooled[c].values) for k, c in cols.items()}).T.round(3))

pv = bt.pivot(index="month", columns="model", values="MAE")
wins = int((pv["LightGBM v3 (+weather)"] < pv["LightGBM v2"]).sum())
print(f"\nv3 (+weather) beats v2 in {wins} of {len(months)} months")

print("\nWet hours vs dry hours (weather helps most when it rains?)")
for label, mask in [("dry", pooled["wet"] == 0), ("wet", pooled["wet"] == 1)]:
    sub = pooled[mask]
    print(label, f"({len(sub)} rows)",
          "| v2 MAE:", scores(sub["trips"].values, sub["pred_v2"].values)["MAE"].round(2),
          "| v3 MAE:", scores(sub["trips"].values, sub["pred_v3"].values)["MAE"].round(2))

bt.to_csv(f"{out}/backtest_metrics.csv", index=False)
pooled.to_parquet(f"{out}/backtest_preds.parquet")

## 13. Bands for the backtest and dashboard export (upgrade)

Fits 10th and 90th percentile models per month, reports 80% band coverage per month, and saves the
files used by the Streamlit dashboard (`app.py`) to the `dashboard_data` folder.
Set `BEST` to `"v2"` if the weather features did not help.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, requests, os

out = "/content/drive/MyDrive/rush-hour-radar"
d3 = pd.read_parquet(f"{out}/features_v3.parquet")
pooled = pd.read_parquet(f"{out}/backtest_preds.parquet")

F_v2 = ["zone", "h", "dow", "month", "is_holiday", "hol_eve", "hol_after", "xmas_period",
        "lag_24", "lag_168", "lag_336", "lag_wk_mean"]
F_v3 = F_v2 + ["temp_c", "precip_mm", "wet"]
months = pd.period_range(pooled["hour"].min().to_period("M"), pooled["hour"].max().to_period("M"), freq="M")

BEST = "v3"        # set to "v2" if the weather features did not help in the backtest above
feats = F_v3 if BEST == "v3" else F_v2

QK = dict(objective="quantile", n_estimators=400, learning_rate=0.05, num_leaves=31,
          min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
          random_state=42, verbose=-1, n_jobs=-1)

parts, cov_rows = [], []
for p in months:
    t0, t1 = p.start_time, (p + 1).start_time
    tr = d3[d3["hour"] < t0]
    te = d3[(d3["hour"] >= t0) & (d3["hour"] < t1)][["hour", "PULocationID"] + feats].copy()
    q10 = lgb.LGBMRegressor(alpha=0.1, **QK).fit(tr[feats], tr["trips"])
    q90 = lgb.LGBMRegressor(alpha=0.9, **QK).fit(tr[feats], tr["trips"])
    te["q10"], te["q90"] = q10.predict(te[feats]), q90.predict(te[feats])
    parts.append(te[["hour", "PULocationID", "q10", "q90"]])
    print(p, "bands done")

bands = pd.concat(parts, ignore_index=True)
res = pooled.merge(bands, on=["hour", "PULocationID"], how="left")
res["pred"] = res[f"pred_{BEST}"]
res["lo"] = np.maximum(0, np.minimum(res["q10"], res["pred"]))
res["hi"] = np.maximum(res["q90"], res["pred"])
res["covered"] = (res["trips"] >= res["lo"]) & (res["trips"] <= res["hi"])

res["width"] = res["hi"] - res["lo"]
cov = (res.assign(month=res["hour"].dt.to_period("M").astype(str))
          .groupby("month").agg(coverage=("covered", "mean"), mean_width=("width", "mean"))
          .round(3))
print(cov)
print("overall coverage (target 0.80):", res["covered"].mean().round(3))

# files for the dashboard
dash = f"{out}/dashboard_data"
os.makedirs(dash, exist_ok=True)
res[["hour", "PULocationID", "trips", "pred", "lo", "hi"]].astype(
    {"trips": "int32", "pred": "float32", "lo": "float32", "hi": "float32"}
).to_parquet(f"{dash}/dashboard_predictions.parquet")
pd.read_csv(f"{out}/backtest_metrics.csv").to_csv(f"{dash}/backtest_metrics.csv", index=False)
cov.reset_index().to_csv(f"{dash}/backtest_coverage.csv", index=False)
try:   # zone names for a friendlier dashboard (optional)
    z = pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
    z.to_csv(f"{dash}/zone_names.csv", index=False)
except Exception as e:
    print("zone names not downloaded (dashboard will show zone IDs):", e)
print(os.listdir(dash))